# All Steps of former Piepline

In [17]:
import requests
import pandas as pd
import io  # Import the io module

# Define the URL
url_crs = "https://sdmx.oecd.org/dcd-public/rest/data/OECD.DCD.FSD,DSD_CRS@DF_CRS,1.3/DAC..1000.100._T._T.D.Q._T..?startPeriod=2019&format=csvfile"

url = 'https://sdmx.oecd.org/public/rest/data/OECD.SDD.NAD,DSD_NAAG@DF_NAAG_I?format=csvfile' 

response = requests.get(url_crs)

print(response.status_code)  # Check if the request was successful

200


In [ ]:
import io
import re
import zipfile
import pandas as pd
import requests

BULK_DOWNLOAD_URL = "https://stats.oecd.org/wbos/fileview2.aspx?IDFile="
BASE_DATAFLOW = "https://sdmx.oecd.org/public/rest/dataflow/OECD.DCD.FSD/"
CRS_FLOW_URL = BASE_DATAFLOW + "DSD_CRS@DF_CRS/"
latest_flow = 1.3

response = requests.get(f"{CRS_FLOW_URL}{latest_flow}")
response.raise_for_status()
content = response.text
search_string="CRS-Parquet"
match = re.search(f"{re.escape(search_string)}(.*?)</", content)
parquet_link = match.group(1).strip()
file_id = parquet_link.split("=")[-1]

file_url = BULK_DOWNLOAD_URL + file_id

file_url

'https://stats.oecd.org/wbos/fileview2.aspx?IDFile=50f0355e-8f61-4230-85f3-90b4db45bfc9'

In [33]:
# Get the file
response = requests.get(file_url, stream=True)
# Check if the request was successful
response.raise_for_status()

save_to_path = None  # Replace with your desired path or None

# Open the content as a zip file and extract the parquet files
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    # Find all parquet files in the zip archive
    parquet_files = [name for name in z.namelist() if name.endswith(".parquet")]

    # If save_to_path is provided, save the files to the path
    if save_to_path:
        save_to_path.mkdir(parents=True, exist_ok=True)
        for file_name in parquet_files:
            with z.open(file_name) as f_in, (save_to_path / file_name).open(
                "wb"
            ) as f_out:
                f_out.write(f_in.read())
    
    files = [pd.read_parquet(z.open(file)) for file in parquet_files]

if files:
    combined_df = pd.concat(files, ignore_index=True)

In [39]:
import sys 

sys.getsizeof(combined_df)

#combined_df.to_csv("../../data/raw/crs_raw.csv", index=False)
combined_df.to_feather("../../data/raw/crs_raw.feather")